<a href="https://colab.research.google.com/github/Saldon1986/Simulative_ML/blob/main/%D0%92%D1%82%D0%BE%D1%80%D0%BE%D0%B9_%D0%BA%D0%B5%D0%B9%D1%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 2 кейс

**Соберите данные из Google Sheets и напишите функцию `generate_report`, которая принимает три списка (данные с каждого листа в Google Sheets) и сохраняет отчет в файл `student_debt_report.txt`**

**Важно**

Перед началом решения задачи выполните следующую ячейку - в ней скачиваются нужные файлы

In [1]:
!wget https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json

!wget https://gist.github.com/Vs8th/39c5deed0f5539d781f00328f7fd4fe0/raw/result.txt

--2026-02-28 18:13:53--  https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.github.com (gist.github.com)... 140.82.113.3
Connecting to gist.github.com (gist.github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json [following]
--2026-02-28 18:13:53--  https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2358 (2.3K) [text/plain]
Saving to: ‘creds.json’

creds.json          100%[===================>]   2.30K  --.-KB/s    in 0s      

2026-02-28 18:13:53 (12.5 MB/s) - ‘creds.json’ saved [2

Чтобы посмотреть как выглядят сами гугл таблицы в оригинале - можете перейти по [ссылке](https://docs.google.com/spreadsheets/d/1hRnw-PEftF0J-6KY7InlILVwWdkJu4vJiGwRjuE_P4U/edit?usp=sharing) и изучить.

Небольшое описание столбцов в таблицах:  
**Лист1:** `student_id` - айди студента; `student_name` - имя студента; `installment` (Y/N) - факт наличия рассрочки у студента (Y - рассрочка есть, N - рассрочки нет).  
**Лист2:** `student_id` - айди студента; `last_payment_date` - дата последней полученной оплаты; `expected_payment_date` - дата, в которую ожидаем следующий платеж (рассчитывается от даты последнего платежа).  
**Лист3:** `student_id` - айди студента; `already_payed_amount` - уже выплаченная часть рассрочки; `left_to_pay` - оставшаяся (невыплаченная) часть; `one-time_payment` - единоразовый платеж; `installment_amount` - общая сумма, которая бралась в рассрочку.

Как примерно должен выглядеть результат:

In [ ]:
with open('result.txt') as f:
  print(f.read())

In [ ]:
#@title Если возникнут трудности с определением `scope`, подсказка:

scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         'https://www.googleapis.com/auth/drive']

**Примечание**

Считать должников необходимо на 1 марта 2023 года. То есть определяя количество просроченных платежей, мы определяем их количество не на настоящую дату, а на **1 марта 2023 года**. А периоды внесения платежей для всех студентов одинаковы - **6 месяцев** (183 дня).

То есть, если ожидаемый платеж должен был пройти 3 февраля 2022 года, то к 1 марта 2023 студент просрочил уже три платежа (3 февраля 2022, 3 августа 2022 и 3 февраля 2023). При расчетах отталкивайтесь от даты ожидаемого платежа, разницу между платежами берите 183 дня.

**Решение**

Напишите свое решение ниже

In [2]:
# блок импорта библиотек
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from datetime import date, datetime, timedelta


In [3]:
# Указываем необходимые права доступа к таблицам
scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         'https://www.googleapis.com/auth/drive']

# Загружаем ключи аутентификации из файла json
creds = ServiceAccountCredentials.from_json_keyfile_name('creds.json', scope)

# Авторизуемся в Google Sheets API
client = gspread.authorize(creds)

# загружаем данные в списки словарей
#Sheet1
sheet = client.open("Installments").worksheet("Лист1")
students = sheet.get_all_records()
#Sheet2
sheet = client.open("Installments").worksheet("Лист2")
p_dates = sheet.get_all_records()
#Sheet3
sheet = client.open("Installments").worksheet("Лист3")
p_details = sheet.get_all_records()

In [4]:
def generate_report(students, p_dates, p_details):
    """
    Функция создания отчета о должниках:
    1) объединяет данные с листов по каждому студенту в один справочник
    2) для студентов с рассрочкой расситывает дни с последнего платежа, периодичность оплаты, кол-во пропущенных платежей на report_date и сумму долга
    3) строит справочник строк
    4) записывает этот справочник в файл student_debt_report.txt
    """
    report_date = date(2023, 3, 1)
    # Создаём словари для быстрого доступа по student_id
    p_dates_dict = {d['student_id']: d for d in p_dates}
    p_details_dict = {d['student_id']: d for d in p_details}

    merged_students=[]

    for student in students:
        student_id = student['student_id']

        # Получаем соответствующие записи (или пустые словари)
        date_record = p_dates_dict.get(student_id, {})
        detail_record = p_details_dict.get(student_id, {})

        # Объединяем
        merged = {**student, **date_record, **detail_record}
        #для студентов с рассрочкой:
        if student['installment'] == 'Y':
        # считаем кол-во пропущенных платежей на отчетную дату
          merged['d_from_lpay'] = (
              report_date -
              datetime.strptime(merged['expected_payment_date'], '%d.%m.%Y').date()).days
          merged['p_missed'] = merged['d_from_lpay'] // 183 + 1
          merged['p_missed_sum'] = min(merged['one-time_payment'] * merged['p_missed'], merged['left_to_pay'])
          #добавляем запись в итоговый список
          if merged['p_missed_sum'] > 0:
            merged_students.append(f'Студент {merged['student_name']} - долг {merged['p_missed_sum']} рублей\n')
    # пишем список в файл
    with open('student_debt_report.txt', 'w', encoding='utf-8') as rep_file:
      rep_file.writelines(merged_students)

    return merged_students


In [5]:
generate_report(students, p_dates, p_details)

['Студент Кузнецова К.А. - долг 266666 рублей\n',
 'Студент Петров К.А. - долг 66666 рублей\n',
 'Студент Иванов А.А. - долг 100000 рублей\n',
 'Студент Иванов П.П. - долг 116666 рублей\n',
 'Студент Петров И.И. - долг 116666 рублей\n',
 'Студент Петров А.А. - долг 16666 рублей\n',
 'Студент Иванов Е.В. - долг 41666 рублей\n',
 'Студент Николаева Е.В. - долг 100000 рублей\n',
 'Студент Смирнова Е.В. - долг 233333 рублей\n',
 'Студент Николаева М.А. - долг 33332 рублей\n',
 'Студент Кузнецова А.А. - долг 133332 рублей\n',
 'Студент Иванов А.А. - долг 25000 рублей\n',
 'Студент Петров И.И. - долг 133332 рублей\n',
 'Студент Кузнецова К.А. - долг 133333 рублей\n',
 'Студент Кузнецова Е.В. - долг 66666 рублей\n',
 'Студент Сидоров И.И. - долг 83333 рублей\n',
 'Студент Петров И.И. - долг 16666 рублей\n',
 'Студент Кузнецова И.И. - долг 233333 рублей\n',
 'Студент Иванов А.А. - долг 266666 рублей\n',
 'Студент Сидоров К.А. - долг 266666 рублей\n',
 'Студент Смирнова И.И. - долг 200000 рубле

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [6]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt

import pandas as pd

user_answer = pd.read_csv('student_debt_report.txt')
correct_answer = pd.read_csv('student_debt.txt')

--2026-02-28 18:14:31--  https://gist.github.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt
Resolving gist.github.com (gist.github.com)... 140.82.114.4
Connecting to gist.github.com (gist.github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt [following]
--2026-02-28 18:14:31--  https://gist.githubusercontent.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11325 (11K) [text/plain]
Saving to: ‘student_debt.txt’

student_debt.txt    100%[===================>]  11.06K  --.-KB/s    in 0s      

2026-02-28 18:14:31 (34.5 MB/s)

In [7]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
  assert user_answer.columns.equals(correct_answer.columns), 'Названия столбцов не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!
